In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u
import astropy.visualization
import named_arrays as na
import msfc_ccd

In [ ]:
astropy.visualization.quantity_support();

In [ ]:
fe55 = msfc_ccd.Fe55()

print(f"K-alpha  {fe55.energy_k_alpha:0.4f}  ->  {fe55.charge_k_alpha:0.1f}")
print(f"K-beta   {fe55.energy_k_beta:0.4f}  ->  {fe55.charge_k_beta:0.1f}")

In [ ]:
warm = msfc_ccd.Fe55(temperature=300 * u.K)

print(f"{fe55.charge_k_alpha:0.1f} at {fe55.temperature}")
print(f"{warm.charge_k_alpha:0.1f} at {warm.temperature}")

In [ ]:
axis_time = "time"

images = msfc_ccd.fits.open(
    path=na.ScalarArray(
        ndarray=np.array(msfc_ccd.samples.paths_fe55_esis3),
        axes=axis_time,
    ),
)

taps = images.taps

axis_x = taps.axis_x
axis_y = taps.axis_y
axis_tap_x = taps.axis_tap_x
axis_tap_y = taps.axis_tap_y

In [ ]:
hits = taps.hits()

num = np.sum(np.isfinite(hits.outputs), axis=(axis_time, axis_x, axis_y))

print(f"events found in each tap, over {taps.shape[axis_time]} images:")
print(num.ndarray)

In [ ]:
axis_charge = "charge"

hist = na.histogram(
    a=hits.outputs,
    bins={axis_charge: 81},
    axis=(axis_time, axis_x, axis_y),
    min=0 * u.DN,
    max=800 * u.DN,
)

edges = hist.inputs
centers = (edges[{axis_charge: slice(None, -1)}] + edges[{axis_charge: slice(1, None)}]) / 2

In [ ]:
fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps.shape[axis_tap_y],
    ncols=taps.shape[axis_tap_x],
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
na.plt.stairs(
    edges,
    hist.outputs,
    axis=axis_charge,
    ax=ax,
    baseline=None,
)
na.plt.set_ylabel("number of events", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("charge (DN)", ax=ax[{axis_tap_y: 0}])
na.plt.text(
    x=0.05,
    y=0.95,
    s=taps.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="left",
    va="top",
);

In [ ]:
tap = {axis_tap_x: 0, axis_tap_y: 0}
peak = centers[np.argmax(hist.outputs[tap], axis=axis_charge)]

# The event in the first image whose charge is closest to the peak
charge = hits.outputs[tap][{axis_time: 0}]
index = np.nanargmin(np.abs(charge - peak))
index = {ax: index[ax].ndarray for ax in index}

half = 4
region = {
    axis_x: slice(index[axis_x] - half, index[axis_x] + half + 1),
    axis_y: slice(index[axis_y] - half, index[axis_y] + half + 1),
}

fig, ax = plt.subplots(figsize=(4, 3.5), constrained_layout=True)
image = na.plt.pcolormesh(
    C=taps.unbiased.active.outputs[tap][{axis_time: 0}][region].value,
    axis_rgb=None,
    ax=ax,
)
ax.set_xlabel("detector $x$ (pix)")
ax.set_ylabel("detector $y$ (pix)")
plt.colorbar(image.ndarray.item(), ax=ax, label="signal (DN)");

In [ ]:
gain = taps.gain().outputs

gain.ndarray

In [ ]:
charge_k_alpha = (fe55.charge_k_alpha / gain).to(u.DN)
charge_k_beta = (fe55.charge_k_beta / gain).to(u.DN)

fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps.shape[axis_tap_y],
    ncols=taps.shape[axis_tap_x],
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
na.plt.stairs(
    edges,
    hist.outputs,
    axis=axis_charge,
    ax=ax,
    baseline=None,
)
na.plt.axvline(
    x=charge_k_alpha,
    ax=ax,
    color="tab:orange",
    linestyle="--",
    label=r"fitted K-$\alpha$",
)
na.plt.axvline(
    x=charge_k_beta,
    ax=ax,
    color="tab:green",
    linestyle="--",
    label=r"fitted K-$\beta$",
)
na.plt.set_ylabel("number of events", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("charge (DN)", ax=ax[{axis_tap_y: 0}])
handles, labels = ax.ndarray.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncols=2);

In [ ]:
published = na.ScalarArray(
    ndarray=np.array([[2.57, 2.59], [2.53, 2.52]]) * u.electron / u.DN,
    axes=(axis_tap_y, axis_tap_x),
)

for i in range(taps.shape[axis_tap_y]):
    for j in range(taps.shape[axis_tap_x]):
        index = {axis_tap_y: i, axis_tap_x: j}
        print(
            f"  {taps.amplifier[index].ndarray}"
            f"   ours {gain[index].ndarray:0.3f}"
            f"   MSFC {published[index].ndarray:0.2f}"
            f"   {100 * (gain[index] / published[index] - 1).ndarray:+5.1f}%"
        )

In [ ]:
flats = msfc_ccd.fits.open(
    path=na.ScalarArray(
        ndarray=np.array([
            msfc_ccd.samples.path_led_esis1,
            msfc_ccd.samples.path_led_esis1_next,
        ]),
        axes=axis_time,
    ),
).taps

darks = msfc_ccd.fits.open(
    path=na.ScalarArray(
        ndarray=np.array([
            msfc_ccd.samples.path_led_dark_esis1,
            msfc_ccd.samples.path_led_dark_esis1_next,
        ]),
        axes=axis_time,
    ),
).taps

In [ ]:
ptc_flats = flats.photon_transfer(axis_time)
ptc_darks = darks.photon_transfer(axis_time)

for i in range(flats.shape[axis_tap_y]):
    for j in range(flats.shape[axis_tap_x]):
        index = {axis_tap_y: i, axis_tap_x: j, axis_time: 0}
        print(
            f"  {flats.amplifier[index].ndarray}"
            f"   flats: signal {ptc_flats.inputs[index].ndarray:7.0f},"
            f" variance {ptc_flats.outputs[index].ndarray:6.0f}"
            f"   darks: variance {ptc_darks.outputs[index].ndarray:4.1f}"
        )

In [ ]:
axis_tile = "tile"
axis_pixel = "pixel"


def tiles(a: na.AbstractScalar, n: int = 32) -> na.ScalarArray:
    """Split each tap into square tiles of n by n pixels."""
    x = a.ndarray_aligned((axis_tap_y, axis_tap_x, axis_y, axis_x))
    num_y = x.shape[-2] // n
    num_x = x.shape[-1] // n
    x = x[..., : num_y * n, : num_x * n]
    x = x.reshape(x.shape[:2] + (num_y, n, num_x, n))
    x = np.moveaxis(x, 4, 3)
    x = x.reshape(x.shape[:2] + (num_y * num_x, n * n))
    return na.ScalarArray(x, axes=(axis_tap_y, axis_tap_x, axis_tile, axis_pixel))


rows = {axis_y: slice(flats.camera.sensor.num_masked, None)}
active = flats.unbiased.active.outputs[rows]
first = active[{axis_time: 0}]
second = active[{axis_time: 1}]

signal_tiles = tiles((first + second) / 2).mean(axis_pixel)
variance_tiles = np.var(tiles(second - first), axis=axis_pixel) / 2

In [ ]:
gain_ptc = flats.gain_photon_transfer(axis_time).outputs

gain_ptc.ndarray

In [ ]:
signal_line = na.linspace(0, 30000, axis="signal", num=2) * u.DN
variance_line = signal_line * u.electron / gain_ptc
variance_line = variance_line + ptc_darks.outputs[{axis_time: 0}]

fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=flats.shape[axis_tap_y],
    ncols=flats.shape[axis_tap_x],
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
na.plt.plot(
    signal_tiles,
    variance_tiles,
    ax=ax,
    axis=axis_tile,
    linestyle="none",
    marker=".",
    markersize=2,
    alpha=0.3,
    label="tiles",
)
na.plt.plot(
    ptc_darks.inputs,
    ptc_darks.outputs,
    ax=ax,
    axis=axis_time,
    linestyle="none",
    marker="o",
    color="black",
    label="darks",
)
na.plt.plot(
    signal_line,
    variance_line,
    ax=ax,
    axis="signal",
    color="tab:red",
    label="fitted gain",
)
na.plt.set_ylabel("variance (DN$^2$)", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("signal (DN)", ax=ax[{axis_tap_y: 0}])
na.plt.text(
    x=0.05,
    y=0.95,
    s=flats.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="left",
    va="top",
)
handles, labels = ax.ndarray.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncols=3);